In [1]:
#library for retreiving data from dotenv file
import os
from dotenv import load_dotenv

#for model building using OpenAi
from openai import OpenAI

load_dotenv()
api_key=os.environ.get("GROQ_API_KEY")

In [2]:
from langchain_core.prompts import PromptTemplate

prompt_temp_name = PromptTemplate(
    input_variables = ['cuisine'],
    template = "I want to open a restaurant for {cuisine} food , suggest me 10 cool names. give just names not any explanation"
)

prompt_temp_name.format(cuisine = "mexican")

'I want to open a restaurant for mexican food , suggest me 10 cool names. give just names not any explanation'

In [3]:
from langchain_core.prompts import ChatPromptTemplate #for prompts 
from langchain_core.output_parsers import StrOutputParser # for structured o/p
from langchain_groq import ChatGroq #llm

#initialize basic llm using llama
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7
)

#create prompt template
chat_prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in simple terms and bullet points , not too long just ion 5 lines."
)


parser = StrOutputParser()

#this is chain using pipe operator : prompt-> llm -> o/p
chain = chat_prompt | llm | parser

response = chain.invoke({"topic" : "Generative AI"})

print(response)

Generative AI creates new content:
* Text
* Images
* Music
* Videos
It uses machine learning to generate unique and original material.


In [4]:
# MultiChaining -> one than one prompts layers

explain_prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in detail."
)

summary_prompt = ChatPromptTemplate.from_template(
    "Summarize this text in 3 bullet points:\n\n{text}"
)

explanation_chain = explain_prompt | llm | parser

summary_chain = summary_prompt | llm | parser

#this is modern sequential chain
full_chain = explanation_chain | summary_chain

result = full_chain.invoke({"topic": "Transformers in AI"})

print(result)

Here are three bullet points summarizing the text:

* Transformers are a type of neural network architecture that relies on self-attention mechanisms to process input sequences, such as text or images, and have revolutionized the field of natural language processing (NLP) since their introduction in 2017.
* The key components of a Transformer include the self-attention mechanism, encoder-decoder architecture, multi-head attention, and positional encoding, which allow the model to weigh the importance of different input elements, preserve the order of the input sequence, and jointly attend to information from different representation subspaces.
* Transformers have achieved state-of-the-art results in many NLP tasks, including machine translation, text classification, and question answering, and have been widely adopted in many other areas of artificial intelligence (AI), with variants such as BERT, RoBERTa, and Transformer-XL further improving their performance and versatility.


In [5]:
# Sequential Chain with multiple o/p
from langchain_core.runnables import RunnableParallel # it helps for multiple outputs at same time in form of dictionary

# chain = {
#     "restaurants": restaurant_chain,
#     "food_items": food_chain
# }

restaurant_prompt = ChatPromptTemplate.from_template(
    "Suggest 5 popular restaurants for {cuisine} cuisine. Give just name not any explanation."
)
restaurant_chain = restaurant_prompt | llm | parser

food_prompt = ChatPromptTemplate.from_template(
    "List 5 famous food items from {cuisine} cuisine. Give just name not any explanation."
)
food_chain = food_prompt | llm | parser

multi_output_chain = RunnableParallel(
    restaurants=restaurant_chain,
    food_items=food_chain
)
result = multi_output_chain.invoke({"cuisine": "Indian"})

print("Restaurants:\n", result["restaurants"])
print("\nFood Items:\n", result["food_items"])

Restaurants:
 1. Tandoori Nights
2. Indian Accent
3. Dhaba
4. Sagar Ratna
5. Karim's

Food Items:
 1. Tandoori Chicken
2. Biryani
3. Naan
4. Samosa
5. Gulab Jamun
